In [45]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles here:")
print(os.listdir()[:30])

Current notebook folder:
C:\Users\vhcha\Untitled Folder 1

Files here:
['.ipynb_checkpoints', 'emotion_recognition_results.csv', 'mel_spectogram.jpeg', 'seesion-4.ipynb', 'Untitled.ipynb']


In [46]:
import os

os.rename("mel_spectogram.jpeg", "mel_spectrogram.jpeg")

print(os.listdir())

['.ipynb_checkpoints', 'emotion_recognition_results.csv', 'mel_spectrogram.jpeg', 'seesion-4.ipynb', 'Untitled.ipynb']


# Technical Report: Multimodal Emotion Recognition using RAVDESS

**Dataset:** RAVDESS Emotional Speech Audio  
**Framework:** TensorFlow / Keras  
**Task:** Multimodal Emotion Recognition using Audio and Text  

---

## 1. Project Overview

The objective of this project is to build a deep learning system that classifies human emotions using two modalities:

1. **Audio modality**: Speech audio is converted into Mel-spectrograms and passed through an audio model.
2. **Text modality**: Transcript text is tokenized, padded, embedded, and passed through a GRU model.
3. **Multimodal fusion**: Feature vectors from both audio and text models are combined to make the final emotion prediction.

The final system compares three models:

- Audio-only model
- Text-only GRU model
- Multimodal early fusion model

---

## 2. Dataset Description

The dataset used is **RAVDESS Emotional Speech Audio**.

RAVDESS contains 1,440 audio clips from 24 actors. Each audio file follows a structured filename format, from which the emotion label can be extracted.

### Emotion Labels

| Code | Emotion |
|---|---|
| 01 | Neutral |
| 02 | Calm |
| 03 | Happy |
| 04 | Sad |
| 05 | Angry |
| 06 | Fearful |
| 07 | Disgust |
| 08 | Surprised |

Example filename:

```text
03-01-05-01-02-02-12.wav
```

In this filename, the third part is `05`, which corresponds to the emotion **angry**.

---

## 3. Dataset Challenges

RAVDESS is useful for emotion recognition, but it also has some important challenges:

| Challenge | Explanation |
|---|---|
| Small dataset | 1,440 clips is small for deep learning, so overfitting can happen. |
| Fixed spoken sentences | The dataset mainly contains two repeated sentences, so the text branch has limited information. |
| Similar emotions | Emotions like calm and neutral can sound very similar. |
| Acted emotions | The emotions are acted, so they may not fully represent real-world spontaneous speech. |
| Audio preprocessing sensitivity | The model performance depends strongly on how audio is converted into features. |

---

## 4. Audio Preprocessing

The raw `.wav` audio files were loaded using Librosa.

Each audio file was converted into a **Mel-spectrogram**. A Mel-spectrogram is a 2D representation of sound that shows how frequency content changes over time.

This is useful because emotions are often expressed through:

- Pitch
- Loudness
- Tone
- Rhythm
- Energy variation

### Audio Preprocessing Pipeline

```text
Raw Audio File
      ↓
Load audio using Librosa
      ↓
Pad / trim to fixed duration
      ↓
Convert to Mel-spectrogram
      ↓
Convert to decibel scale
      ↓
Standardize values
      ↓
Feed into audio model
```

A sample Mel-spectrogram was plotted in the notebook to visually confirm the audio preprocessing step.

![Sample Mel-Spectrogram](mel_spectrogram.jpeg)

---

## 5. Text Generation and Preprocessing

For the text branch, transcripts were generated or inferred from the known RAVDESS spoken statements.

RAVDESS mainly uses two fixed statements:

1. “Kids are talking by the door”
2. “Dogs are sitting by the door”

The transcript was processed using the following steps:

```text
Transcript
      ↓
Tokenization
      ↓
Padding
      ↓
Embedding Layer
      ↓
GRU Layer
      ↓
Emotion Prediction
```

Since the same sentences are repeated across emotions, the text branch is expected to be weaker than the audio branch. This is because the actual words do not strongly indicate the emotion.

---

## 6. Model Architectures

## 6.1 Audio-only Model

The audio model uses audio features extracted from speech. The main idea is to learn patterns related to vocal emotion, such as pitch, tone, loudness, and rhythm.

### Audio Architecture Diagram

```text
Audio File
   ↓
Mel-Spectrogram / Audio Feature Extraction
   ↓
Audio Neural Network
   ↓
Dense Bottleneck Layer
   ↓
Softmax Output Layer
   ↓
Predicted Emotion
```

### Design Decision

The audio branch is important because emotional information is mainly present in the way the sentence is spoken. Even if the words are the same, the tone, pitch, intensity, and rhythm change based on emotion.

---

## 6.2 Text-only GRU Model

The text model uses the generated transcript as input.

### Text Architecture Diagram

```text
Transcript
   ↓
Tokenization
   ↓
Padding
   ↓
Embedding Layer
   ↓
GRU Layer
   ↓
Dense Bottleneck Layer
   ↓
Softmax Output Layer
   ↓
Predicted Emotion
```

### Design Decision

A GRU was used because text is sequential data. GRU can process word sequences and learn simple patterns from the transcript.

GRU was chosen instead of a larger text model because the dataset has only two main spoken sentences, so a lightweight model is enough.

---

## 6.3 Multimodal Early Fusion Model

The multimodal model combines both audio and text information.

### Multimodal Architecture Diagram

```text
                 AUDIO BRANCH
        Audio → Feature Extraction → Audio Bottleneck
                            ↓
                       Concatenation
                            ↑
                 TEXT BRANCH
        Transcript → Embedding → GRU → Text Bottleneck

                       Concatenated Vector
                            ↓
                       Dense Layers
                            ↓
                       Softmax Output
                            ↓
                       Predicted Emotion
```

### Design Decision

Early fusion was used because it allows the model to combine information from both modalities before making the final prediction.

The audio branch captures **how** the sentence is spoken.  
The text branch captures **what** is spoken.  
The fusion model combines both feature vectors and predicts the final emotion.

---

## 7. Loss Function and Metrics

### Loss Function

The loss function used was **Categorical Cross-Entropy** because this is a multi-class classification problem with 8 emotion classes.

### Evaluation Metrics

The models were evaluated using:

1. Accuracy
2. Weighted F1-score
3. Classification report
4. Confusion matrix
5. Training and validation loss plots

Accuracy measures overall correctness. Weighted F1-score is useful because it considers precision and recall across all emotion classes.

---

## 8. Results

The final comparison of the models is shown below.



![Final Results Table](results_table.jpeg)

---

## 9. Confusion Matrix Analysis

A confusion matrix was plotted for the multimodal fusion model.

The confusion matrix helps us understand which emotions were predicted correctly and which emotions were confused with each other.

Possible confusion patterns:

- Calm and neutral may be confused because both can have low intensity.
- Sad and calm may overlap because both may sound soft and slow.
- Happy and surprised may be confused because both may have higher pitch and energy.
- Fearful and surprised may also overlap due to high vocal intensity.

![Confusion Matrix](confusion_matrix.jpeg)

---

## 10. Training and Validation Loss Plots

Training and validation loss plots were generated for:

1. Audio-only model
2. Text-only GRU model
3. Multimodal fusion model

These plots help analyze whether the model is learning properly or overfitting.

If training loss decreases while validation loss increases, it indicates overfitting or overconfidence on wrong validation samples. This can happen in small datasets like RAVDESS.

<img src="./audio_loss.jpeg" alt="Audio Training and Validation Loss" width="700">

![Text Training and Validation Loss](text_loss.jpeg)

![Fusion Training and Validation Loss](fusion_loss.jpeg)

---

## 11. Analysis of Results

The audio-only model achieved around 63–65% accuracy, which is meaningful because the task has 8 emotion classes. Random guessing would give around 12.5% accuracy.

The audio model performed better because emotional information is mainly present in vocal properties such as:

- Pitch
- Loudness
- Tone
- Rhythm
- Energy

The text-only model is expected to perform weaker because RAVDESS uses mostly fixed spoken sentences. Since the same words are repeated across emotions, the transcript itself does not strongly reveal the emotion.

The multimodal fusion model combines both audio and text features. However, because the text modality is weak in this dataset, the fusion model may perform close to the audio-only model rather than showing a very large improvement.

---

## 12. Challenges Faced

The main challenges faced during the project were:

1. The dataset is small for deep learning.
2. Some emotions have very similar vocal patterns.
3. The text modality is weak because the spoken sentences are mostly fixed.
4. Audio preprocessing strongly affects model performance.
5. Deep learning models can overfit on small datasets.
6. Validation loss may increase even when training accuracy improves because the model becomes overconfident.
7. Multimodal fusion is only highly useful when both modalities contain strong information.

---

## 13. Conclusion

This project implemented a complete multimodal emotion recognition pipeline using the RAVDESS dataset.

Three models were trained and compared:

1. Audio-only model
2. Text-only GRU model
3. Multimodal early fusion model

The models were evaluated using accuracy, weighted F1-score, confusion matrix, classification report, and training-validation loss plots.

The results show that the audio modality is more useful than the text modality for this dataset. This is because RAVDESS emotions are mainly expressed through vocal tone, pitch, loudness, rhythm, and energy, while the transcript content is mostly fixed.

The final system successfully demonstrates a complete multimodal emotion recognition pipeline.